## Results generalization 

Testing the model performance on alternative sequencing runs

In [28]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
import pandas as pd

In [4]:
master_db = pd.read_csv(r"F:\HelixWorks\Basecalling\data\motifcaller_2024-10-07_07-49\master_db.csv")

### Filtering fast5 reads to extract

Extract those reads which have an associated encoded part of the file

In [16]:
filtered_db = master_db.loc[~master_db['HW_Address'].str.startswith('unclassified')]

In [21]:
filenames_db = set(filtered_db['filename'])
read_ids_db = set(filtered_db['read_id'])

### Loading data from fast5 files

In [29]:
from data_functions import get_data_from_fast5
import os
from tqdm.notebook import tqdm

In [6]:
# Selecting reads which have a barcode and are the right file
fast5_filepath = r"F:\HelixWorks\Basecalling\data\motifcaller_2024-10-07_07-49\FAST5"

In [ ]:
master_squiggles = []
master_read_ids = []

In [46]:

for fast5_file in tqdm(os.listdir(fast5_filepath)[20:]):
    squiggles_file, read_ids_file = get_data_from_fast5(
        os.path.join(fast5_filepath, fast5_file), read_ids_db)
    master_squiggles.extend(squiggles_file)
    master_read_ids.extend(read_ids_file)

  0%|          | 0/1111 [00:00<?, ?it/s]

In [47]:
df = pd.DataFrame({"squiggle": master_squiggles, "read_id": master_read_ids})

In [48]:
merged_df = pd.merge(filtered_db, df, on='read_id')

In [ ]:
merged_df.to_pickle(r"C:\Users\Parv\Doc\HelixWorks\Basecalling\code\motifcaller\data\empirical\01-04run\master.pkl")

In [4]:
t = pd.read_pickle(r"C:\Users\Parv\Doc\HelixWorks\Basecalling\code\motifcaller\data\empirical\01-04run\master.pkl")

### Adding reference information

In [14]:
encoded_df = pd.read_csv(r"C:\Users\Parv\Doc\HelixWorks\Basecalling\code\motifcaller\data\empirical\01-04run\HELIX01-04-encoded.csv")

In [15]:
# Joining payloads
payload_cols = [col for col in encoded_df.columns if col.startswith('Payload')]
encoded_df['payload'] = encoded_df[payload_cols].astype(str).agg(', '.join, axis=1)
encoded_df['payload'] = encoded_df['payload'].apply(lambda x: list(eval(x)))

# Fixing addresses
encoded_df['Address_Incrementer_1'] = encoded_df['Address_Incrementer_1'].apply(lambda x: f'barcode_external0{x[1]}')
encoded_df['Address_Incrementer_2'] = encoded_df['Address_Incrementer_2'].apply(lambda x: f'_internal0{x[1]}')
address_cols = [col for col in encoded_df.columns if col.startswith('Address')]
encoded_df['HW_Address'] = encoded_df[address_cols].astype(str).agg(''.join, axis=1)

# Selecting important columns
encoded_df = encoded_df[['HW_Address', 'payload']]

In [16]:
merged_df = pd.merge(t, encoded_df, on='HW_Address')

In [105]:
merged_df.to_pickle(r'C:\Users\Parv\Doc\HelixWorks\Basecalling\code\motifcaller\data\empirical\1-04_small.pkl')

### Mini for barcodes testing

In [21]:
import random

In [32]:
selected_barcodes = random.sample(list(merged_df['HW_Address'].unique()), 5)

In [33]:
filtered_df = merged_df.loc[merged_df['HW_Address'].isin(selected_barcodes)]

In [35]:
filtered_df.to_pickle(r"C:\Users\Parv\Doc\HelixWorks\Basecalling\code\motifcaller\data\empirical\01-04run\filtered_barcodes_testing.pkl")

In [37]:
merged_df.to_pickle(r"C:\Users\Parv\Doc\HelixWorks\Basecalling\code\motifcaller\data\empirical\01-04run\master.pkl")

#